# SHADOWLINE: Combine-to-Game Movement Intelligence for Defensive Backs
### NFL Big Data Bowl 2027 — Combine sensor tracking and regular-season NFL game performance

**Question.** Do pre-draft *movement-quality* characteristics measured by 10 Hz Combine sensor tracking — how efficiently a defensive back decelerates, re-orients and re-accelerates through a direction change — carry information about NFL coverage performance *beyond* traditional Combine measurements (stopwatch drills)?

**Approach.** We introduce the **Mirror Response Index (MRI)**, a family of five kinematic components extracted from Combine drill tracking:

| # | Component | Units | Definition |
|---|-----------|-------|------------|
| 1 | Reorientation latency | s | Time from the plant (speed minimum) to recovery of 70% of entry speed |
| 2 | Turn-speed retention | ratio | Speed retained through the plant vs entry speed |
| 3 | Directional transition efficiency | ratio | Net displacement / path length across the turn window |
| 4 | Exit acceleration | yd/s² | Mean tangential acceleration in the 0.5 s after the plant |
| 5 | Attempt consistency | ratio | 1 − CV across a player's attempts of the same drill |

**Design (pre-registered).** Primary hypothesis: among defensive backs with ≥ 20 regular-season coverage opportunities, MRI is associated with lower mean separation allowed at pass-forward, adjusted for traditional Combine measures. Primary outcome: exposure-weighted mean `separation_at_pass_forward`. Models A (traditional) / B (+MRI) / C (+context) are compared out-of-sample with player-grouped cross-validation, player-clustered bootstrap CIs, within-player permutation tests, leave-one-draft-cohort-out validation and BH-FDR control over secondary analyses.

**Honesty guarantees.** Every number in this notebook either (a) derives from the competition datasets, or (b) is an explicitly-labelled synthetic-fixture validation of an algorithm. If the competition input is not attached, the notebook audits, reports what is missing, and **refuses to produce any statistical claim**.

## 0. Environment, data resolution and the data-presence gate

In [ ]:
import os, sys, json, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

# --- Kaggle dependency policy -------------------------------------------
# Kaggle's image ships numpy/pandas/scipy/matplotlib. We additionally use
# polars + duckdb for fast, memory-bounded I/O (both ship with Kaggle's
# python image; the guarded install below covers older images).
try:
    import polars as pl
    import duckdb
except ImportError:  # pragma: no cover
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "polars", "duckdb"],
                   check=True)
    import polars as pl
    import duckdb
import scipy.stats as sps

print(f"numpy {np.__version__} | pandas {pd.__version__} | polars {pl.__version__} | duckdb {duckdb.__version__}")

In [ ]:
EXPECTED_FILES = [
    "players.csv", "combine_results.csv", "combine_tracking.csv",
    "player_career_successes.csv", "games.csv", "player_play.csv",
    "game_tracking_2023.csv", "game_tracking_2024.csv", "game_tracking_2025.csv",
]

def resolve_data_dir():
    cands = [
        Path("/kaggle/input/nfl-big-data-bowl-2027"),
        Path("/kaggle/input/nfl-big-data-bowl-2027/NFL Big Data Bowl 2027"),
        Path("data/raw"),
        Path("."),
    ]
    for c in cands:
        if c.exists() and any(c.glob("players.csv")):
            return c
    for c in Path("/kaggle/input").glob("*") if Path("/kaggle/input").exists() else []:
        if any(c.rglob("players.csv")):
            return next(iter(c.rglob("players.csv"))).parent
    return cands[0]

DATA_DIR = resolve_data_dir()
print(f"Data directory: {DATA_DIR}")

FILE_AUDIT = {}
for f in EXPECTED_FILES:
    p = DATA_DIR / f
    if p.exists():
        h = hashlib.sha256()
        with open(p, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                h.update(chunk)
        n_rows = sum(1 for _ in open(p, "rb")) - 1
        cols = open(p, encoding="utf-8", errors="replace").readline().strip().split(",")
        FILE_AUDIT[f] = {"present": True, "rows": n_rows, "sha256": h.hexdigest()[:16],
                         "columns": [c.strip() for c in cols]}
    else:
        FILE_AUDIT[f] = {"present": False}

DATA_PRESENT = all(v["present"] for v in FILE_AUDIT.values())
print(pd.DataFrame([{ "file": f, "present": v["present"],
                      "rows": v.get("rows"), "sha256_16": v.get("sha256")}
                    for f, v in FILE_AUDIT.items()]).to_string(index=False))
print(f"\nDATA_PRESENT = {DATA_PRESENT}")
if not DATA_PRESENT:
    print("\n" + "=" * 70 +
          "\nCOMPETITION DATA NOT FULLY PRESENT. This notebook will run the\n"
          "algorithm parity tests (synthetic fixtures, clearly labelled) and\n"
          "the data audit, but NO analysis, figure or statistical claim that\n"
          "requires the competition datasets will be produced. Attach the\n"
          "NFL Big Data Bowl 2027 dataset to this notebook and re-run.\n" + "=" * 70)

## 1. Algorithm embed (verbatim from the SHADOWLINE pipeline)

The cells below are the exact numpy-pure algorithm modules of the
[`shadowline`](https://github.com/) pipeline (see `shadowline/kinematics.py`,
`shadowline/mri.py`, `shadowline/synth.py`), embedded verbatim with only
relative-import lines stripped. A SHA-256 tag of each embed is printed so any
drift from the repository source is detectable.

In [ ]:
# --- VERBATIM EMBED: shadowline/kinematics.py (sha256:dd78d4961147ac94) ---
# Relative imports stripped; all definitions live in this notebook's
# namespace. Parity-tested against the repository unit-test fixtures
# in the cell below. Do not edit by hand: regenerate with
# scripts/build_notebook.py
"""Kinematics: circular angle math, derivatives, smoothing, reversal detection.

This module is deliberately dependency-light (numpy only) so it can be
embedded verbatim in the standalone Kaggle notebook (see
scripts/build_notebook.py, which copies this file's source into the notebook
and runs a parity test against the same synthetic fixtures used here).

Conventions
-----------
* Angles are in DEGREES on the NFL convention: 0 degrees points along +x,
  90 along +y (values in [0, 360)).
* Heading = direction of the velocity vector, computed from (dx, dy).
* All derivative functions are sampling-frequency aware (dt per sample).
* Smoothing NEVER overwrites raw measurements: smoothed values are returned
  alongside; downstream consumers choose.
"""
from __future__ import annotations

import numpy as np

EPS = 1e-9


# ---------------------------------------------------------------------------
# Circular math
# ---------------------------------------------------------------------------
def wrap_deg(angle: float | np.ndarray) -> float | np.ndarray:
    """Wrap degrees to [0, 360). Handles negative and >360 inputs."""
    return np.mod(np.asarray(angle, dtype=float), 360.0)


def angular_diff(a: float | np.ndarray, b: float | np.ndarray) -> float | np.ndarray:
    """Smallest absolute angular difference between two headings, degrees.

    Correct across the 0/360 boundary: angular_diff(350, 10) == 20.
    """
    d = np.abs(wrap_deg(a) - wrap_deg(b)) % 360.0
    return np.minimum(d, 360.0 - d)


def circular_mean(angles: np.ndarray, weights: np.ndarray | None = None) -> float:
    """Circular mean via atan2 of mean sin/cos. Returns degrees in [0, 360)."""
    r = np.deg2rad(np.asarray(angles, dtype=float))
    if weights is None:
        s, c = np.sin(r).mean(), np.cos(r).mean()
    else:
        w = np.asarray(weights, dtype=float)
        s, c = (np.sin(r) * w).sum() / w.sum(), (np.cos(r) * w).sum() / w.sum()
    if abs(s) < EPS and abs(c) < EPS:
        return float("nan")
    deg = float(np.rad2deg(np.arctan2(s, c)) % 360.0)
    return 0.0 if deg > 360.0 - 1e-9 else deg  # guard 360.0 == 0.0 edge


def unwrap_deg(angles: np.ndarray) -> np.ndarray:
    """Continuous (unwrapped) version of a degree sequence, radians-free API."""
    return np.rad2deg(np.unwrap(np.deg2rad(wrap_deg(angles).astype(float))))


def heading(dx: np.ndarray, dy: np.ndarray) -> np.ndarray:
    """Movement heading in degrees [0, 360) from displacement components."""
    return wrap_deg(np.rad2deg(np.arctan2(dy, dx)))


def bearing(fx: np.ndarray, fy: np.ndarray, px: np.ndarray, py: np.ndarray) -> np.ndarray:
    """Bearing (degrees) from focus point (fx, fy) toward point (px, py)."""
    return heading(px - fx, py - fy)


# ---------------------------------------------------------------------------
# Derivatives (sampling-aware, central differences in the interior)
# ---------------------------------------------------------------------------
def time_steps(t: np.ndarray) -> np.ndarray:
    """Forward time differences; assumes t strictly increasing (validated by
    the quality module before this is used)."""
    return np.diff(np.asarray(t, dtype=float))


def speed_from_positions(x: np.ndarray, y: np.ndarray, dt: np.ndarray | float) -> np.ndarray:
    """Speed (yd/s) from consecutive positions. len == len(x)-1."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    step = np.hypot(np.diff(x), np.diff(y))
    return step / dt if np.isscalar(dt) or np.ndim(dt) == 0 else step / dt


def accel_from_speed(s: np.ndarray, dt: np.ndarray | float) -> np.ndarray:
    """Tangential acceleration (yd/s^2), central differences in the interior."""
    s = np.asarray(s, float)
    ds = np.diff(s)
    if np.ndim(dt) != 0:
        ds = ds / dt
    else:
        ds = ds / float(dt)
    a = np.empty(len(s))
    a[1:-1] = (ds[1:] + ds[:-1]) / 2.0
    a[0], a[-1] = ds[0], ds[-1]
    return a


def path_length(x: np.ndarray, y: np.ndarray) -> float:
    """Total travelled distance from consecutive coordinates (yd).

    Used to RECONSTRUCT distance when the supplied `dis` field is unreliable:
    consumers should prefer this over the raw field after the quality module
    has validated `dis` against the same quantity.
    """
    return float(np.hypot(np.diff(np.asarray(x, float)),
                          np.diff(np.asarray(y, float))).sum())


def cumulative_distance(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Cumulative travelled distance, aligned to frames (len == len(x))."""
    steps = np.hypot(np.diff(np.asarray(x, float)), np.diff(np.asarray(y, float)))
    return np.concatenate([[0.0], np.cumsum(steps)])


# ---------------------------------------------------------------------------
# Smoothing (originals always preserved)
# ---------------------------------------------------------------------------
def moving_average(values: np.ndarray, window: int) -> np.ndarray:
    """Centered moving average with edge mirroring. window odd, >=1."""
    values = np.asarray(values, float)
    if window <= 1:
        return values.copy()
    if window % 2 == 0:
        window += 1
    pad = window // 2
    padded = np.pad(values, pad, mode="reflect")
    kernel = np.ones(window) / window
    return np.convolve(padded, kernel, mode="valid")


def smooth_heading(heading_deg: np.ndarray, window: int) -> np.ndarray:
    """Smooth an angular series correctly: unwrap -> smooth -> re-wrap."""
    unwrapped = unwrap_deg(heading_deg)
    return wrap_deg(moving_average(unwrapped, window))


# ---------------------------------------------------------------------------
# Reversal (direction-change) detection
# ---------------------------------------------------------------------------
def detect_reversals(
    heading_deg: np.ndarray,
    speed: np.ndarray,
    dt: float,
    *,
    heading_window_s: float = 0.5,
    min_event_heading_change_deg: float = 60.0,
    min_turn_angle_deg: float = 90.0,
    merge_events_within_s: float = 0.4,
    min_speed_for_heading: float = 0.5,
    smoothing_window: int = 5,
) -> list[dict]:
    """Detect direction-reversal events in a trajectory.

    Method
    ------
    1. Unwrap + smooth the heading series (circularly correct).
    2. Compute cumulative absolute heading change in a rolling window;
       flag frames where it exceeds ``min_event_heading_change_deg``.
    3. Within each candidate event, the reversal centre is the frame of
       minimum smoothed speed (biomechanical "plant").
    4. Accept the event if the NET heading change across the event window
       exceeds ``min_turn_angle_deg``.
    5. Merge events whose centres are within ``merge_events_within_s``.

    Returns a list of event dicts with indices and measured turn angle.
    """
    n = len(heading_deg)
    if n < 5:
        return []
    hs = smooth_heading(heading_deg, smoothing_window)
    ss = moving_average(speed, smoothing_window)
    unwrapped = unwrap_deg(hs)
    dtheta = np.diff(unwrapped)  # deg per frame

    w = max(1, int(round(heading_window_s / dt)))
    # rolling sum of |dtheta| over w frames
    kernel = np.ones(w)
    cum_change = np.convolve(np.abs(dtheta), kernel, mode="valid")  # len n-w
    # net change over the same window
    net_change = np.convolve(dtheta, kernel, mode="valid")

    # candidate frames: index i in cum_change corresponds to window ending at frame i+w
    hot = np.where(cum_change >= min_event_heading_change_deg)[0]

    events: list[dict] = []
    used = np.zeros(n, dtype=bool)
    for i in hot:
        centre = int(i + w // 2)
        if centre <= 0 or centre >= n - 1 or used[centre]:
            continue
        lo = max(0, centre - w)
        hi = min(n - 1, centre + w)
        # plant = min smoothed speed in the neighbourhood
        c_local = int(np.argmin(ss[lo:hi + 1])) + lo
        net = unwrapped[min(c_local + w, n - 1)] - unwrapped[max(c_local - w, 0)]
        if abs(net) < min_turn_angle_deg:
            continue
        used[max(0, c_local - w):min(n, c_local + w + 1)] = True
        events.append({
            "centre_idx": int(c_local),
            "start_idx": int(max(0, c_local - w)),
            "end_idx": int(min(n - 1, c_local + w)),
            "turn_angle_deg": float(abs(net)),
            "turn_sign": int(np.sign(net)),
        })

    # merge close events
    merged: list[dict] = []
    for ev in sorted(events, key=lambda e: e["centre_idx"]):
        if merged and (ev["centre_idx"] - merged[-1]["centre_idx"]) * dt < merge_events_within_s:
            if ev["turn_angle_deg"] > merged[-1]["turn_angle_deg"]:
                merged[-1] = ev
        else:
            merged.append(ev)
    return merged

In [ ]:
# --- VERBATIM EMBED: shadowline/mri.py (sha256:0dc4e1b28f94f600) ---
# Relative imports stripped; all definitions live in this notebook's
# namespace. Parity-tested against the repository unit-test fixtures
# in the cell below. Do not edit by hand: regenerate with
# scripts/build_notebook.py
"""Mirror Response Index (MRI): change-of-direction movement quality metric.

MRI quantifies how efficiently an athlete decelerates, reorients and
re-accelerates during direction reversals, measured from 10 Hz position
tracking of Combine drills. It is designed for defensive backs whose coverage
job is to "mirror" a receiver: absorb a route break, re-orient the hips and
body, and exit the break without losing separation.

Components (per reversal event, aggregated per attempt / player)
---------------------------------------------------------------
1. reorientation_latency (RL)     [s]     Time from the speed minimum (plant)
   to recovery of ``recovery_threshold`` x entry speed.
2. turn_speed_retention (TSR)     [-]     Speed retained through the plant
   relative to the entry speed (v_min / v_entry).
3. directional_transition_efficiency (DTE) [-]  Net displacement over path
   length across the turn window (0, 1]; 1.0 = no wasted mileage.
4. exit_acceleration (EA)         [yd/s^2] Mean tangential acceleration after
   the plant over ``exit_window_s``.
5. attempt_consistency (CONS)     [-]     1 - CV across a player's attempts of
   the same drill; requires >= 2 attempts.

Composite: robust z-scores (median/MAD) of RL (sign-flipped), TSR, DTE, EA
within position x draft-cohort, weighted by split-half reliability when
estimable; requires >= ``min_components`` components; otherwise null.
Higher MRI = better change-of-direction quality.

IMPORTANT (research honesty rules):
* Each component is tested independently before the composite.
* The composite is NOT assumed valid because components can be computed.
* Sensitivity to every parameter is reported (see sensitivity_scan).
"""
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np


DEFAULTS = {
    "heading_window_s": 0.5,
    "min_event_heading_change_deg": 60.0,
    "min_turn_angle_deg": 90.0,
    "merge_events_within_s": 0.4,
    "entry_band_s": (0.8, 0.2),
    "local_min_band_s": 0.3,
    "recovery_threshold": 0.7,
    "recovery_max_wait_s": 1.5,
    "path_window_s": 0.5,
    "exit_window_s": 0.5,
    "min_speed_for_heading_yd_s": 0.5,
    "smoothing_window_frames": 5,
}

COMPONENTS = ("reorientation_latency", "turn_speed_retention",
              "directional_transition_efficiency", "exit_acceleration")


@dataclass
class AttemptFeatures:
    """MRI features for one drill attempt."""
    nfl_id: int | None = None
    drill: str = ""
    attempt: int | None = None
    n_reversals: int = 0
    reorientation_latency: float | None = None
    turn_speed_retention: float | None = None
    directional_transition_efficiency: float | None = None
    exit_acceleration: float | None = None
    attempt_time_s: float | None = None
    path_length_yd: float | None = None
    mean_speed_yd_s: float | None = None
    peak_speed_yd_s: float | None = None
    quality_flags: list[str] = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "nfl_id": self.nfl_id, "drill": self.drill, "attempt": self.attempt,
            "n_reversals": self.n_reversals,
            "reorientation_latency": self.reorientation_latency,
            "turn_speed_retention": self.turn_speed_retention,
            "directional_transition_efficiency": self.directional_transition_efficiency,
            "exit_acceleration": self.exit_acceleration,
            "attempt_time_s": self.attempt_time_s,
            "path_length_yd": self.path_length_yd,
            "mean_speed_yd_s": self.mean_speed_yd_s,
            "peak_speed_yd_s": self.peak_speed_yd_s,
            "quality_flags": self.quality_flags,
        }


def _null_features(reason: str, **kwargs) -> AttemptFeatures:
    f = AttemptFeatures(**kwargs)
    f.quality_flags.append(reason)
    return f


def attempt_components(
    x: np.ndarray, y: np.ndarray, dt: float, *,
    heading_deg: np.ndarray | None = None,
    speed: np.ndarray | None = None,
    params: dict | None = None,
) -> AttemptFeatures:
    """Compute MRI components for ONE attempt (frame-aligned arrays).

    Missing-data behaviour: any component that cannot be validly measured is
    None (never imputed); the reason is recorded in quality_flags.
    """
    p = {**DEFAULTS, **(params or {})}
    x = np.asarray(x, float); y = np.asarray(y, float)
    n = len(x)
    if n < 10:
        return _null_features("too_few_frames", attempt_time_s=n * dt)

    # Frame-level kinematics (reconstructed from positions; raw s/a are kept
    # by the caller for cross-validation in the quality stage).
    if speed is None:
        speed = np.concatenate([[0.0], np.hypot(np.diff(x), np.diff(y)) / dt])
    if heading_deg is None:
        dx, dy = np.diff(x), np.diff(y)
        spd_step = np.hypot(dx, dy) / dt
        hd = np.empty(n)
        hd[0] = heading(dx[0], dy[0]) if spd_step[0] > p["min_speed_for_heading_yd_s"] else np.nan
        hd[1:] = np.where(spd_step > p["min_speed_for_heading_yd_s"],
                          wrap_deg(np.rad2deg(np.arctan2(dy, dx))), np.nan)
        # forward-fill undefined headings (stationary frames)
        idx = np.where(np.isfinite(hd))[0]
        if len(idx) < 3:
            return _null_features("heading_undefined", attempt_time_s=n * dt)
        hd = np.where(np.isfinite(hd), hd, hd[idx[0]])
        heading_deg = hd

    if speed is None or len(speed) != n:
        speed = np.concatenate([[0.0], np.hypot(np.diff(x), np.diff(y)) / dt])

    sm = moving_average(speed, p["smoothing_window_frames"])

    out = AttemptFeatures(
        attempt_time_s=n * dt,
        path_length_yd=path_length(x, y),
        mean_speed_yd_s=float(np.nanmean(speed)),
        peak_speed_yd_s=float(np.nanmax(sm)),
    )

    events = detect_reversals(
        heading_deg, speed, dt,
        heading_window_s=p["heading_window_s"],
        min_event_heading_change_deg=p["min_event_heading_change_deg"],
        min_turn_angle_deg=p["min_turn_angle_deg"],
        merge_events_within_s=p["merge_events_within_s"],
        min_speed_for_heading=p["min_speed_for_heading_yd_s"],
        smoothing_window=p["smoothing_window_frames"],
    )
    out.n_reversals = len(events)
    if not events:
        out.quality_flags.append("no_reversal_detected")
        return out

    rl_list, tsr_list, dte_list, ea_list = [], [], [], []
    for ev in events:
        c = ev["centre_idx"]
        # entry speed: robust speed in the entry band before the plant
        e_lo = max(0, int(c - p["entry_band_s"][0] / dt))
        e_hi = max(0, int(c - p["entry_band_s"][1] / dt))
        entry_band = sm[e_lo:e_hi] if e_hi > e_lo else sm[max(0, c - 1):c]
        v_entry = float(np.quantile(entry_band, 0.9)) if len(entry_band) else float(sm[max(0, c - 1)])

        # local minimum speed around the plant
        m_lo = max(0, int(c - p["local_min_band_s"] / dt))
        m_hi = min(n, int(c + p["local_min_band_s"] / dt) + 1)
        v_min = float(np.min(sm[m_lo:m_hi]))

        # 1. reorientation latency: time from plant until speed >= threshold*v_entry
        if v_entry > 0:
            thr = p["recovery_threshold"] * v_entry
            max_wait = int(p["recovery_max_wait_s"] / dt)
            rec_idx = None
            for j in range(c, min(n, c + max_wait)):
                if sm[j] >= thr:
                    rec_idx = j
                    break
            if rec_idx is not None:
                rl_list.append((rec_idx - c) * dt)
            # 2. turn-speed retention
            tsr_list.append(v_min / v_entry)

        # 3. DTE: net displacement / path length across the turn window
        p_lo = max(0, int(c - p["path_window_s"] / dt))
        p_hi = min(n - 1, int(c + p["path_window_s"] / dt))
        if p_hi > p_lo:
            net = float(np.hypot(x[p_hi] - x[p_lo], y[p_hi] - y[p_lo]))
            plen = float(cumulative_distance(x[p_lo:p_hi + 1], y[p_lo:p_hi + 1])[-1])
            if plen > 1e-6:
                dte_list.append(net / plen)

        # 4. exit acceleration: mean tangential accel after the plant
        x_hi = min(n, int(c + p["exit_window_s"] / dt) + 1)
        if x_hi > c + 1:
            acc = accel_from_speed(sm[c:x_hi], dt)
            ea_list.append(float(np.clip(np.mean(acc), -30.0, 30.0)))

    if rl_list:
        out.reorientation_latency = float(np.mean(rl_list))
    else:
        out.quality_flags.append("latency_unmeasured")
    if tsr_list:
        out.turn_speed_retention = float(np.mean(tsr_list))
    else:
        out.quality_flags.append("retention_unmeasured")
    if dte_list:
        out.directional_transition_efficiency = float(np.mean(dte_list))
    else:
        out.quality_flags.append("dte_unmeasured")
    if ea_list:
        out.exit_acceleration = float(np.mean(ea_list))
    else:
        out.quality_flags.append("exit_accel_unmeasured")

    # validity checks
    if out.reorientation_latency is not None and not (0.0 <= out.reorientation_latency <= p["recovery_max_wait_s"]):
        out.reorientation_latency = None
        out.quality_flags.append("latency_out_of_range")
    if out.turn_speed_retention is not None and not (0.0 <= out.turn_speed_retention <= 1.5):
        out.turn_speed_retention = None
        out.quality_flags.append("retention_out_of_range")
    if out.directional_transition_efficiency is not None and not (0.0 < out.directional_transition_efficiency <= 1.0):
        out.directional_transition_efficiency = None
        out.quality_flags.append("dte_out_of_range")
    return out


# ---------------------------------------------------------------------------
# Consistency + composite
# ---------------------------------------------------------------------------
def attempt_consistency(values: list[float], min_attempts: int = 2) -> float | None:
    """CONS = 1 - CV (coefficient of variation) across attempts. None if <2."""
    v = np.asarray([x for x in values if x is not None and np.isfinite(x)], float)
    if len(v) < min_attempts:
        return None
    mu = float(v.mean())
    if mu <= 1e-9:
        return None
    return float(1.0 - v.std(ddof=1) / mu)


def robust_z(values: np.ndarray) -> np.ndarray:
    """Median/MAD robust standardisation (outlier-resistant)."""
    v = np.asarray(values, float)
    if not np.isfinite(v).any():
        return np.full_like(v, np.nan)
    med = np.nanmedian(v)
    mad = np.nanmedian(np.abs(v - med))
    if mad < 1e-9:
        sd = np.nanstd(v)
        if sd < 1e-9:
            return np.zeros_like(v)
        return (v - med) / (1.4826 * sd)
    return (v - med) / (1.4826 * mad)


def mri_composite(
    rows: list[dict],
    *,
    reliability_weights: dict[str, float] | None = None,
    min_components: int = 3,
) -> list[dict]:
    """Compute the MRI composite for a cohort of players.

    ``rows``: list of dicts with keys nfl_id and any of the component names
    (player-level aggregated values). Returns rows with 'mri_composite' and
    'mri_components_used'. Higher = better. Latency is sign-flipped (lower
    latency is better). Cohort standardisation happens INSIDE this function:
    callers pass a single position x draft-year cohort.
    """
    comp_cols = ["reorientation_latency", "turn_speed_retention",
                 "directional_transition_efficiency", "exit_acceleration"]
    flip = {"reorientation_latency": -1.0}
    w = reliability_weights or {}

    def _lookup(row: dict, c: str) -> float:
        """Accept both bare and aggregated ('_mean') key variants."""
        for k in (c, f"{c}_mean"):
            v = row.get(k)
            if v is not None and isinstance(v, (int, float)) and np.isfinite(v):
                return float(v)
        return np.nan

    matrix = {c: np.array([_lookup(r, c) for r in rows], float) for c in comp_cols}
    z = {c: robust_z(matrix[c]) * flip.get(c, 1.0) for c in comp_cols}

    out = []
    for i, r in enumerate(rows):
        num, den, used = 0.0, 0.0, []
        for c in comp_cols:
            if np.isfinite(z[c][i]):
                weight = float(w.get(c, 1.0))
                num += z[c][i] * weight
                den += weight
                used.append(c)
        row = dict(r)
        row["mri_components_used"] = used
        row["mri_composite"] = float(num / den) if len(used) >= min_components else None
        out.append(row)
    return out


def sensitivity_scan(x: np.ndarray, y: np.ndarray, dt: float, grid: dict) -> dict:
    """Re-run component extraction over a parameter grid; report value ranges.

    Used to demonstrate that components are not knife-edge artifacts of the
    chosen parameters (Phase 3 sensitivity requirement).
    """
    results = {}
    keys = list(grid.keys())
    import itertools
    combos = list(itertools.product(*[grid[k] for k in keys]))
    for combo in combos:
        params = dict(zip(keys, combo))
        f = attempt_components(x, y, dt, params=params)
        for c in COMPONENTS:
            v = getattr(f, c)
            if v is not None:
                results.setdefault(c, []).append(round(v, 4))
    summary = {}
    for c, vals in results.items():
        arr = np.asarray(vals, float)
        summary[c] = {
            "n_parameter_combinations": len(vals),
            "min": float(arr.min()), "max": float(arr.max()),
            "median": float(np.median(arr)),
            "max_relative_spread": float((arr.max() - arr.min()) / max(abs(np.median(arr)), 1e-9)),
        }
    return summary


# ---------------------------------------------------------------------------
# Game-side mirror behaviour (shared with coverage.py)
# ---------------------------------------------------------------------------
def mirror_latency(
    defender_x: np.ndarray, defender_y: np.ndarray,
    receiver_x: np.ndarray, receiver_y: np.ndarray,
    event_idx: int, dt: float, *,
    error_threshold_deg: float = 30.0,
    sustain_s: float = 0.3,
    min_speed: float = 0.5,
    cap_s: float = 2.0,
) -> float | None:
    """Time after a receiver direction change until the defender is re-oriented
    toward the receiver: bearing-to-receiver vs defender heading error below
    ``error_threshold_deg`` sustained for ``sustain_s``. None if never."""
    n = min(len(defender_x), len(receiver_x))
    if event_idx >= n - 2:
        return None
    sustain = max(1, int(sustain_s / dt))
    cap = int(cap_s / dt)
    streak = 0
    for j in range(event_idx, min(n, event_idx + cap)):
        dx, dy = np.diff(defender_x[j:j + 2])[0] if j + 1 < n else 0.0, \
                 np.diff(defender_y[j:j + 2])[0] if j + 1 < n else 0.0
        spd = np.hypot(dx, dy) / dt
        if spd < min_speed:
            streak = 0
            continue
        hd = heading(np.array([dx]), np.array([dy]))[0]
        br = heading(np.array([receiver_x[j] - defender_x[j]]),
                     np.array([receiver_y[j] - defender_y[j]]))[0]
        if angular_diff(hd, br) <= error_threshold_deg:
            streak += 1
            if streak >= sustain:
                return (j - event_idx) * dt
        else:
            streak = 0
    return None

In [ ]:
# --- VERBATIM EMBED: shadowline/synth.py (sha256:9d9cb48d81b9212c) ---
# Relative imports stripped; all definitions live in this notebook's
# namespace. Parity-tested against the repository unit-test fixtures
# in the cell below. Do not edit by hand: regenerate with
# scripts/build_notebook.py
"""Deterministic synthetic fixtures for TESTS and the labelled algorithm demo.

RULES (research honesty):
* These fixtures are used ONLY to (a) unit-test mathematical functions against
  known ground truth, (b) run the integration pipeline end-to-end in CI, and
  (c) power a clearly-labelled "algorithm demonstration" in the Tracking
  Laboratory UI. They are NEVER used as competition evidence and never
  presented as NFL results. Every artifact that contains synthetic data is
  marked with ``synthetic: true``.
"""
from __future__ import annotations

import numpy as np

DEMO_SEED = 2027
DEMO_LABEL = "SYNTHETIC DEMONSTRATION FIXTURE — NOT NFL DATA"


def smoothstep_vel(profile_t: np.ndarray, dt: float, rng: np.random.Generator,
                   noise: float = 0.0) -> np.ndarray:
    raise NotImplementedError  # (kept minimal; not used)


def synthetic_shuttle_attempt(
    *,
    dt: float = 0.1,
    start_speed: float = 7.0,
    plant_speed: float = 1.2,
    exit_speed: float = 6.0,
    noise_m: float = 0.01,
    seed: int = DEMO_SEED,
) -> dict:
    """A 20-yard short-shuttle-like trajectory with TWO 180-degree reversals.

    Geometry (yards): start at (0, 5), sprint +x for 10 yd, 180 deg reversal,
    sprint -x for 10 yd, second reversal, sprint +x for 10 yd. Speeds dip to
    ``plant_speed`` at each reversal. Ground truth component values are
    computed analytically where possible.

    Returns dict with arrays x, y, t, speed, heading and ground-truth metadata.
    """
    rng = np.random.default_rng(seed)

    def leg(v0: float, v1: float, v2: float, length: float, direction: int):
        """One leg: accelerate v0->v2, decelerate into the plant at v1."""
        # simple trapezoidal profile in distance
        n_acc = max(2, int(0.4 / dt))
        n_dec = max(2, int(0.5 / dt))
        speeds = np.concatenate([
            np.linspace(v0, v2, n_acc),
            np.full(2, v2),
            np.linspace(v2, v1, n_dec),
        ])
        dists = speeds * dt
        # scale to exactly `length` by stretching the cruise portion
        total = dists.sum()
        if total > 0:
            n_cruise = max(1, int(round((length - 0.0) / (v2 * dt))))
            cruise = np.full(n_cruise, v2)
            all_spd = np.concatenate([np.linspace(v0, v2, n_acc), cruise,
                                      np.linspace(v2, v1, n_dec)])
            d = all_spd * dt
            d = d * (length / d.sum())
            return all_spd, d
        return speeds, dists

    legs = [
        leg(start_speed, plant_speed, 6.5, 10.0, +1),
        leg(plant_speed, plant_speed, exit_speed, 10.0, -1),
        leg(exit_speed, plant_speed, 6.5, 10.0, +1),
    ]
    speeds = np.concatenate([l[0] for l in legs])
    seg_dirs = np.concatenate([np.full(len(legs[0][0]), +1.0),
                               np.full(len(legs[1][0]), -1.0),
                               np.full(len(legs[2][0]), +1.0)])
    dists = np.concatenate([l[1] for l in legs])

    x = np.concatenate([[0.0], np.cumsum(dists * seg_dirs)])
    x = x + rng.normal(0, noise_m, size=len(x))
    y = np.full_like(x, 5.0) + rng.normal(0, noise_m, size=len(x))
    t = np.arange(len(x)) * dt

    step_speed = np.hypot(np.diff(x), np.diff(y)) / dt
    speed = np.concatenate([[0.0], step_speed])
    hd = np.full(len(x), 0.0)
    hd[1:] = np.where(np.diff(x) >= 0, 0.0, 180.0)
    hd = wrap_deg(hd)

    # ground truth: reversals at the two plants (speed minima away from edges)
    interior_minima = [i for i in range(3, len(speed) - 3)
                       if speed[i] == speed[max(0, i - 2):i + 3].min()
                       and speed[i] < 3.0]
    # keep the two deepest minima (one per interior plant)
    gt_plants = sorted(interior_minima, key=lambda i: speed[i])[:2]

    return {
        "label": DEMO_LABEL,
        "synthetic": True,
        "x": x.tolist(), "y": y.tolist(), "t": t.tolist(),
        "speed": speed.tolist(), "heading": hd.tolist(),
        "dt": dt,
        "ground_truth": {
            "n_reversals": 2,
            "plant_indices": gt_plants,
            "plant_speed": plant_speed,
            "exit_speed": exit_speed,
        },
        "description": (
            "Synthetic short-shuttle-style trajectory: two 180-degree reversals "
            "with known plant speeds, generated with a fixed random seed. Used "
            "for algorithm validation and UI demonstration only."
        ),
    }


def synthetic_coverage_play(*, dt: float = 0.1, seed: int = 4242) -> dict:
    """Synthetic coverage rep: a defender mirroring a receiver that breaks out.

    Receiver runs +x 8 yd then breaks 90 deg toward +y. Defender starts 3 yd
    inside (offset -y) and must re-orient. Ground truth: separation at pass
    forward ~ known; mirror latency measurable.
    """
    rng = np.random.default_rng(seed)
    n1 = 55  # frames before break
    n2 = 40  # frames after break
    t = np.arange(n1 + n2 + 1) * dt

    # receiver: straight upfield then break toward +y at frame n1
    rx = np.concatenate([np.linspace(0, 8.0, n1), np.full(n2 + 1, 8.0)])
    ry = np.concatenate([np.full(n1, 5.0), np.linspace(5.0, 10.0, n2 + 1)])

    # defender: mirrors, offset 2.5 yd inside, reacts with a delay
    delay = 6  # frames of reaction delay (0.6 s)
    dx = np.concatenate([
        np.linspace(0, 7.4, n1),
        np.linspace(7.4, 8.4, 12),
        np.linspace(8.4, 8.4, n2 - 11),
    ])[:n1 + n2 + 1]
    dy = np.concatenate([
        np.full(n1, 2.5),
        np.full(len(t) - n1, 2.5),
    ])
    # after receiver break, defender closes toward the receiver's new path
    idx = np.arange(n1, len(t))
    dy[idx] = 2.5 + np.clip((idx - n1 - delay) * 0.06, 0, 3.4)

    rx = rx + rng.normal(0, 0.005, len(t))
    ry = ry + rng.normal(0, 0.005, len(t))
    dx = dx + rng.normal(0, 0.005, len(t))
    dy = dy + rng.normal(0, 0.005, len(t))

    sep = np.hypot(rx - dx, ry - dy)
    return {
        "label": DEMO_LABEL,
        "synthetic": True,
        "t": t.tolist(), "dt": dt,
        "receiver": {"x": rx.tolist(), "y": ry.tolist()},
        "defender": {"x": dx.tolist(), "y": dy.tolist()},
        "separation": sep.tolist(),
        "break_index": n1,
        "pass_forward_index": n1 + 25,   # 2.5 s after snap-ish; arbitrary demo
        "ground_truth": {
            "receiver_break_frame": n1,
            "defender_reaction_delay_s": delay * dt,
            "separation_at_pass_forward": float(sep[n1 + 25]),
        },
        "description": (
            "Synthetic coverage rep (out-break route vs off-coverage corner) "
            "with a known reaction delay. Used for algorithm validation and "
            "UI demonstration only."
        ),
    }


def synthetic_player_level_dataset(n_players: int = 60, *, effect: float = -0.35,
                                   seed: int = 99) -> dict:
    """Synthetic player-level dataset with a KNOWN effect for statistical
    framework testing (Phase 5 tests). Never used as evidence.

    Returns dict of arrays: mri, forty, shuttle, separation (outcome). The
    outcome is generated as: sep = a + effect*mri + b1*forty + noise, so the
    true partial effect of MRI (given forty) is ``effect``.
    """
    rng = np.random.default_rng(seed)
    mri = rng.normal(0, 1, n_players)
    forty = rng.normal(4.55, 0.12, n_players)
    shuttle = 4.35 + 0.1 * mri + rng.normal(0, 0.15, n_players)  # correlated with mri
    noise = rng.normal(0, 0.6, n_players)
    separation = 3.0 + effect * mri + 1.5 * (forty - 4.55) + noise
    return {
        "n_players": n_players, "mri": mri, "forty": forty,
        "shuttle": shuttle, "separation": separation,
        "true_mri_partial_effect": effect,
        "synthetic": True,
    }

In [ ]:
# --- VERBATIM EMBED: shadowline/quality.py (sha256:7544474ef53fb605) ---
# Relative imports stripped; all definitions live in this notebook's
# namespace. Parity-tested against the repository unit-test fixtures
# in the cell below. Do not edit by hand: regenerate with
# scripts/build_notebook.py
"""Tracking data-quality subsystem (Phase 2).

Checks (each parameterised, machine-readable, per attempt / per player-play):
  sampling_interval_consistency   -- dt distribution vs expected 0.1 s (10 Hz)
  missing_frames                  -- frame gaps (delta frame > 1)
  repeated_frames                 -- duplicate frame ids / timestamps
  position_discontinuity          -- step distance implausible vs speed bound
  impossible_velocity_change      -- |ds| beyond accel bound * dt
  coordinate_validity             -- field bounds (game) / envelope (combine)
  angle_validity                  -- o/dir within [0, 360), NaN share
  short_attempts                  -- attempts/plays below minimum frames
  entity_types                    -- distinct tracked entities (player/BALL)
  drill_label_audit               -- canonicalisation coverage of drill labels
  cohort_measurement_differences  -- KS tests of dt/position noise by cohort

Also validates the supplied ``dis`` field against distance reconstructed
from consecutive x/y (the field is unreliable in some vendor feeds) and
produces the recommendation used downstream (reconstructed distance).
"""
from __future__ import annotations

import numpy as np
import polars as pl
from scipy import stats as sps

TOL_DT = 0.02  # +/- 20 ms tolerance on the 100 ms interval


def dt_stats(t: np.ndarray) -> dict:
    t = np.asarray(t, float)
    if len(t) < 2:
        return {"n": 0}
    dt = np.diff(t)
    good = dt[dt > 0]
    return {
        "n": int(len(dt)),
        "median_s": float(np.median(good)) if len(good) else None,
        "p01_s": float(np.quantile(good, 0.01)) if len(good) else None,
        "p99_s": float(np.quantile(good, 0.99)) if len(good) else None,
        "share_within_tolerance": float(np.mean(np.abs(good - EXPECTED_DT) <= TOL_DT)) if len(good) else None,
        "n_zero_or_negative": int(np.sum(dt <= 0)),
    }


def frame_gap_stats(frame: np.ndarray) -> dict:
    frame = np.asarray(frame, float)
    if len(frame) < 2:
        return {"n_gaps": 0, "largest_gap": 0, "n_repeated": 0}
    d = np.diff(frame)
    gaps = d[d > 1]
    return {
        "n_gaps": int(len(gaps)),
        "largest_gap": int(gaps.max()) if len(gaps) else 0,
        "n_repeated": int(np.sum(d == 0)),
        "n_frames": int(len(frame)),
        "n_expected_frames": int(frame[-1] - frame[0] + 1) if len(frame) else 0,
    }


def discontinuity_stats(x: np.ndarray, y: np.ndarray, t: np.ndarray,
                        supplied_s: np.ndarray | None = None) -> dict:
    """Position continuity checks + speed/accel plausibility."""
    x, y, t = map(lambda a: np.asarray(a, float), (x, y, t))
    step = np.hypot(np.diff(x), np.diff(y))
    dt = np.maximum(np.diff(t), EXPECTED_DT / 2)
    implied_speed = step / dt
    flags = {
        "n_steps": int(len(step)),
        "n_over_speed_bound": int(np.sum(implied_speed > MAX_SPEED_YD_S)),
        "max_implied_speed_yd_s": float(np.max(implied_speed)) if len(implied_speed) else None,
        "n_teleport_steps": int(np.sum(step > MAX_SPEED_YD_S * dt)),  # same bound, explicit
    }
    if supplied_s is not None:
        s = np.asarray(supplied_s, float)
        if len(s) == len(x):
            ds = np.abs(np.diff(s))
            flags["n_impossible_accel"] = int(np.sum(ds > MAX_ACCEL_YD_S2 * dt))
            # distance field vs reconstructed distance
            recon = np.diff(cumulative_distance(x, y))
            flags["dis_vs_reconstructed_mae_yd"] = float(np.mean(np.abs(recon - step))) if len(step) else None
    return flags


def coordinate_bounds(x: np.ndarray, y: np.ndarray, domain: str = "game") -> dict:
    x, y = np.asarray(x, float), np.asarray(y, float)
    if domain == "game":
        out = {
            "x_out_of_field": int(np.sum((x < 0) | (x > NFL_FIELD_LENGTH_YD))),
            "y_out_of_field": int(np.sum((y < 0) | (y > NFL_FIELD_WIDTH_YD))),
        }
    else:  # combine: envelope learned from data (documented per drill later)
        out = {
            "x_min": float(np.nanmin(x)) if len(x) else None,
            "x_max": float(np.nanmax(x)) if len(x) else None,
            "y_min": float(np.nanmin(y)) if len(y) else None,
            "y_max": float(np.nanmax(y)) if len(y) else None,
        }
    out["n_nan_xy"] = int(np.sum(~(np.isfinite(x) & np.isfinite(y))))
    return out


def angle_validity(angles: np.ndarray | None) -> dict | None:
    if angles is None:
        return None
    a = np.asarray(angles, float)
    return {
        "n": int(len(a)),
        "n_nan": int(np.sum(~np.isfinite(a))),
        "n_out_of_0_360": int(np.sum((a < 0) | (a > 360))),
    }


def ks_cohort_test(samples_by_cohort: dict[str, np.ndarray], feature: str) -> dict:
    """Pairwise Kolmogorov-Smirnov tests of a DQ feature across draft cohorts.

    Warns before pooling cohorts when distributions differ materially.
    """
    keys = sorted(samples_by_cohort)
    out = {"feature": feature, "pairs": []}
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            a, b = samples_by_cohort[keys[i]], samples_by_cohort[keys[j]]
            a, b = a[np.isfinite(a)], b[np.isfinite(b)]
            if len(a) < 5 or len(b) < 5:
                continue
            stat, p = sps.ks_2samp(a, b)
            out["pairs"].append({
                "cohorts": [keys[i], keys[j]],
                "ks_stat": float(stat), "p_value": float(p),
                "materially_different": bool(p < 0.01 and stat > 0.15),
            })
    return out


# ---------------------------------------------------------------------------
# Attempt-level quality verdict (used to gate feature extraction)
# ---------------------------------------------------------------------------
def attempt_quality_verdict(x: np.ndarray, y: np.ndarray, t: np.ndarray,
                            frame: np.ndarray | None = None,
                            supplied_s: np.ndarray | None = None,
                            supplied_dis: np.ndarray | None = None,
                            domain: str = "combine",
                            min_frames: int = 15) -> dict:
    """Aggregate all checks into one attempt-level verdict + flags."""
    x, y, t = (np.asarray(a, float) for a in (x, y, t))
    flags: list[str] = []
    dts = dt_stats(t)
    gaps = frame_gap_stats(frame if frame is not None else np.arange(len(t)))
    disc = discontinuity_stats(x, y, t, supplied_s)
    bounds = coordinate_bounds(x, y, domain)

    if len(t) < min_frames:
        flags.append("short_attempt")
    if dts.get("share_within_tolerance", 1.0) is not None and dts["share_within_tolerance"] < 0.9:
        flags.append("irregular_sampling")
    if gaps.get("n_repeated", 0) > 0:
        flags.append("repeated_frames")
    if gaps.get("largest_gap", 0) >= 5:
        flags.append("missing_frames")
    if len(t) > 2:
        t_gaps = np.diff(t)
        if np.any(t_gaps > 3 * EXPECTED_DT):  # > 0.3 s hole at 10 Hz
            if "missing_frames" not in flags:
                flags.append("missing_frames")
    if disc["n_over_speed_bound"] > 0:
        flags.append("implausible_speed")
    if disc.get("n_impossible_accel", 0) > 0:
        flags.append("impossible_accel")
    if domain == "game" and (bounds["x_out_of_field"] > 0 or bounds["y_out_of_field"] > 0):
        flags.append("out_of_field")

    # distance-field reliability: prefer reconstruction when MAE is material
    use_reconstructed = True
    if supplied_dis is not None and len(supplied_dis) == len(x):
        dis = np.asarray(supplied_dis, float)[1:]
        recon = np.diff(cumulative_distance(x, y))
        mae = float(np.mean(np.abs(dis - recon))) if len(dis) else None
        use_reconstructed = mae is None or mae > 0.05
        if mae is not None and mae > 0.05:
            flags.append("dis_field_unreliable")

    return {
        "n_frames": int(len(t)),
        "duration_s": float(t[-1] - t[0]) if len(t) > 1 else 0.0,
        "sampling": dts, "gaps": gaps, "discontinuities": disc, "bounds": bounds,
        "flags": flags,
        "usable": "short_attempt" not in flags
                  and "out_of_field" not in flags
                  and "irregular_sampling" not in flags,
        "use_reconstructed_distance": bool(use_reconstructed),
    }


def quality_report_from_attempts(attempt_rows: list[dict]) -> dict:
    """Aggregate per-attempt verdicts into the machine-readable report."""
    n = len(attempt_rows)
    if n == 0:
        return {"status": "no_attempts", "n_attempts": 0}
    all_flags: dict[str, int] = {}
    for r in attempt_rows:
        for f in r.get("flags", []):
            all_flags[f] = all_flags.get(f, 0) + 1
    usable = sum(1 for r in attempt_rows if r.get("usable"))
    share = usable / n if n else None
    return {
        "status": "ok",
        "n_attempts": n,
        "n_usable": usable,
        "share_usable": round(share, 4) if share is not None else None,
        "flag_counts": dict(sorted(all_flags.items(), key=lambda kv: -kv[1])),
        "reconstructed_distance_recommended": bool(
            sum(1 for r in attempt_rows if r.get("use_reconstructed_distance")) >= max(1, n // 2)),
        "attempts_flagged": [
            {"nfl_id": r.get("nfl_id"), "drill": r.get("drill"), "attempt": r.get("attempt"),
             "flags": r.get("flags", [])}
            for r in attempt_rows if r.get("flags")
        ][:500],  # cap for artifact size; full data stays in parquet
    }

### 1.1 Parity test (runs in every environment, synthetic fixtures only)

These are the *same* ground-truth fixtures used by the repository's unit
test-suite (`tests/test_kinematics.py`, `tests/test_mri.py`). They assert the
embedded algorithms are mathematically correct **before** they touch any real
data — including the circular-angle edge cases (0°/360° boundary) and
change-of-direction detection on a synthetic shuttle with known plant points.

In [ ]:
def _parity_tests():
    # --- circular math: the classic 0/360 boundary bug -----------------
    assert abs(angular_diff(350, 10) - 20) < 1e-9, "angular_diff boundary failure"
    assert abs(circular_mean(np.array([350.0, 10.0])) - 0.0) < 0.1
    # --- derivatives ---------------------------------------------------
    s = speed_from_positions(np.array([0., 1., 2., 3.]), np.zeros(4), 0.1)
    assert np.allclose(s, 10.0)
    assert np.allclose(accel_from_speed(np.array([0., 1., 2., 3.]), 0.1), 10.0)
    # --- reversal detection on synthetic shuttle ------------------------
    fx = synthetic_shuttle_attempt()
    x, y = np.array(fx["x"]), np.array(fx["y"])
    hd = np.array(fx["heading"]); sp = np.array(fx["speed"])
    events = detect_reversals(hd, sp, fx["dt"])
    assert len(events) == 2, f"expected 2 reversals, found {len(events)}"
    for e in events:
        assert 150 <= e["turn_angle_deg"] <= 210
    # --- MRI components on synthetic ground truth ----------------------
    f = attempt_components(x, y, fx["dt"])
    assert f.n_reversals == 2
    assert 0.05 <= f.reorientation_latency <= 1.0
    assert 0.0 < f.turn_speed_retention <= 0.6
    assert 0.0 < f.directional_transition_efficiency <= 1.0
    assert f.exit_acceleration > 0.5
    # --- game-side mirror latency --------------------------------------
    cp = synthetic_coverage_play()
    ml = mirror_latency(np.array(cp["defender"]["x"]), np.array(cp["defender"]["y"]),
                        np.array(cp["receiver"]["x"]), np.array(cp["receiver"]["y"]),
                        cp["break_index"], cp["dt"])
    assert ml is not None and ml >= 0.3
    return True

print("PARITY TESTS:", "PASS" if _parity_tests() else "FAIL")
print("Embedded algorithms validated on synthetic ground truth (NOT competition evidence).")

In [ ]:
# Pre-registered research parameters + domain constants
# (mirrors shadowline/config.py; the notebook is self-contained)
RP = {"alpha": 0.05, "n_bootstrap": 1000, "n_permutation": 5000,
      "cv_folds": 5, "random_seed": 20271106,
      "min_attempts_for_consistency": 2, "min_components_for_composite": 3,
      "min_coverage_plays": 20, "open_threshold_yd": 3.0}
MRI_ELIGIBLE_DRILLS = {"short_shuttle", "long_shuttle", "three_cone", "position_drill"}
# Drill canonicalisation mirrors shadowline/ingest.py: the 2027 package uses
# SCREAMING_SNAKE_CASE labels (SHORT_SHUTTLE, THREE_CONE_DRILL, FORTY_YARD_DASH).
CANONICAL_DRILLS = {
    "40_yard_dash": "forty", "forty": "forty", "40": "forty", "forty_yard_dash": "forty",
    "three_cone": "three_cone", "3_cone": "three_cone", "three_cone_drill": "three_cone",
    "20_yard_shuttle": "short_shuttle", "short_shuttle": "short_shuttle",
    "shuttle": "short_shuttle", "5_10_5": "short_shuttle",
    "60_yard_shuttle": "long_shuttle", "long_shuttle": "long_shuttle",
    "broad_jump": "broad_jump", "vertical_jump": "vertical_jump",
    "bench_press": "bench_press", "bench_reps": "bench_press",
    "position_drill": "position_drill", "positional_drill": "position_drill",
    "on_campus": "on_campus_workout", "pro_day": "pro_day_workout",
}
# domain constants used by the embedded quality module
SAMPLING_HZ = 10.0; EXPECTED_DT = 1.0 / SAMPLING_HZ
MAX_SPEED_YD_S = 12.0; MAX_ACCEL_YD_S2 = 15.0
NFL_FIELD_LENGTH_YD = 120.0; NFL_FIELD_WIDTH_YD = 53.3
DRAFT_COHORTS = [2023, 2024, 2025]

import re  # noqa: E402

def canonical_drill_nb(raw):
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return None
    key = re.sub(r"[\s\-]+", "_", str(raw).strip().lower())
    key = re.sub(r"[^a-z0-9_]", "", key)
    if key in CANONICAL_DRILLS:
        return CANONICAL_DRILLS[key]
    for k, v in CANONICAL_DRILLS.items():
        if key.startswith(k):
            return v
    return key or None

In [ ]:
# --- VERBATIM EMBED (subset): shadowline/research.py statistics ---
# sha256:d8e232445bd77b90 (relative-import stripped; RP defined above)
def spearman(x, y) -> dict:
    x, y = _pair(x, y)
    if len(x) < 4:
        return {"rho": None, "p": None, "n": len(x)}
    rho, p = sps.spearmanr(x, y)
    return {"rho": float(rho), "p": float(p), "n": int(len(x))}


def partial_spearman(x, y, covars: np.ndarray) -> dict:
    """Partial Spearman: rank-transform, residualise on covariates (OLS),
    then Spearman of residuals."""
    x, y = _pair(x, y)
    covars = np.asarray(covars, float)
    if covars.ndim == 1:
        covars = covars[:, None]
    m = np.isfinite(x) & np.isfinite(y) & np.all(np.isfinite(covars), axis=1)
    x, y, covars = x[m], y[m], covars[m]
    n = len(x)
    if n < max(8, covars.shape[1] + 4):
        return {"rho": None, "p": None, "n": n, "note": "insufficient_n"}
    rx, ry = sps.rankdata(x), sps.rankdata(y)
    X = np.column_stack([np.ones(n), covars])
    bx = np.linalg.lstsq(X, rx, rcond=None)[0]
    by = np.linalg.lstsq(X, ry, rcond=None)[0]
    resx, resy = rx - X @ bx, ry - X @ by
    rho, p = sps.spearmanr(resx, resy)
    return {"rho": float(rho), "p": float(p), "n": int(n)}


def clustered_bootstrap_ci(fn, groups: np.ndarray, n_boot: int | None = None,
                           seed: int | None = None, alpha: float = 0.05) -> dict:
    """Cluster (player-level) resampling CI for an arbitrary statistic fn."""
    n_boot = n_boot or RP["n_bootstrap"]
    seed = RP["random_seed"] if seed is None else seed
    rng = np.random.default_rng(seed)
    groups = np.asarray(groups)
    uniq = np.unique(groups)
    stats = []
    for _ in range(n_boot):
        picked = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([np.where(groups == g)[0] for g in picked])
        try:
            v = fn(idx)
            if v is not None and np.isfinite(v):
                stats.append(v)
        except Exception:
            continue
    if not stats:
        return {"ci_low": None, "ci_high": None, "n_success": 0}
    arr = np.asarray(stats, float)
    return {"ci_low": float(np.quantile(arr, alpha / 2)),
            "ci_high": float(np.quantile(arr, 1 - alpha / 2)),
            "n_success": int(len(arr)), "boot_mean": float(arr.mean())}


def permutation_test(stat_fn, x: np.ndarray, y: np.ndarray,
                     n_perm: int | None = None, seed: int | None = None) -> dict:
    """Two-sided permutation test.

    ``stat_fn(x_arr, y_arr) -> scalar``. Under the null, ``x`` is shuffled
    RELATIVE to ``y`` (and any covariates captured in the closure stay
    aligned with ``y``). This is the correct Mantel-style permutation; simply
    permuting a shared index array would be a no-op (a bug class caught by
    the statistical unit tests).
    """
    n_perm = n_perm or RP["n_permutation"]
    seed = RP["random_seed"] if seed is None else seed
    rng = np.random.default_rng(seed)
    x = np.asarray(x, float)
    obs = stat_fn(x, y)
    if obs is None or not np.isfinite(obs):
        return {"observed": None, "p": None}
    count = 0
    for _ in range(n_perm):
        v = stat_fn(rng.permutation(x), y)
        if v is not None and np.isfinite(v) and abs(v) >= abs(obs):
            count += 1
    return {"observed": float(obs), "p": (count + 1) / (n_perm + 1),
            "n_perm": n_perm, "two_sided": True}


def bh_fdr(pvalues: dict[str, float]) -> dict[str, dict]:
    """Benjamini-Hochberg adjusted q-values."""
    keys = [k for k, v in pvalues.items() if v is not None and np.isfinite(v)]
    if not keys:
        return {}
    ps = np.array([pvalues[k] for k in keys])
    order = np.argsort(ps)
    m = len(ps)
    q = np.empty(m)
    prev = 1.0
    for rank_i in range(m - 1, -1, -1):
        i = order[rank_i]
        val = min(prev, ps[i] * m / (rank_i + 1))
        q[i] = val
        prev = val
    return {k: {"p": float(pvalues[k]), "q": float(q[i])}
            for i, k in enumerate(keys)}


def _oos_spearman(y, pred):
    if len(y) < 4 or np.std(pred) < 1e-12:
        return None
    return float(sps.spearmanr(y, pred).statistic)


def cross_validated_oos(y: np.ndarray, X: np.ndarray, player_ids: np.ndarray,
                        folds: int | None = None, seed: int | None = None) -> dict:
    """Grouped K-fold OOS: players never split across train/test. Uses
    ridge-regularised linear regression (small-n discipline)."""
    from sklearn.linear_model import Ridge
    from sklearn.model_selection import GroupKFold
    folds = folds or min(RP["cv_folds"], len(np.unique(player_ids)))
    y = np.asarray(y, float)
    X = np.asarray(X, float)
    if len(y) < folds + 2:
        return {"oos_spearman": None, "n": int(len(y)), "note": "insufficient_n"}
    gkf = GroupKFold(n_splits=folds)
    preds = np.full(len(y), np.nan)
    for tr, te in gkf.split(X, y, groups=player_ids):
        assert set(player_ids[tr]).isdisjoint(set(player_ids[te])), "player leakage in CV split"
        mu, sd = X[tr].mean(0), X[tr].std(0)
        sd[sd < 1e-9] = 1.0
        Z = (X[tr] - mu) / sd
        model = Ridge(alpha=1.0).fit(Z, y[tr])
        preds[te] = model.predict((X[te] - mu) / sd)
    return {"oos_spearman": _oos_spearman(y, preds), "n": int(len(y)),
            "folds": int(folds), "grouping": "nfl_id"}


def leave_one_cohort_out(y: np.ndarray, X: np.ndarray, cohorts: np.ndarray) -> dict:
    """LDOCO: train on two draft cohorts, test on the held-out one."""
    y, X, cohorts = np.asarray(y, float), np.asarray(X, float), np.asarray(cohorts)
    from sklearn.linear_model import Ridge
    results = []
    for c in np.unique(cohorts):
        te = cohorts == c
        tr = ~te
        if te.sum() < 3 or tr.sum() < 5:
            continue
        mu, sd = X[tr].mean(0), X[tr].std(0)
        sd[sd < 1e-9] = 1.0
        m = Ridge(alpha=1.0).fit((X[tr] - mu) / sd, y[tr])
        pred = m.predict((X[te] - mu) / sd)
        results.append({"held_out_cohort": str(c), "n_test": int(te.sum()),
                        "oos_spearman": _oos_spearman(y[te], pred)})
    return {"folds": results}


def assert_no_leakage(feature_cols: list[str], outcome_cols: list[str]) -> None:
    """Hard assertion: no feature may be derived from the outcome."""
    outcome_tokens = {"sep", "separation", "open", "targeted", "exposure_within",
                      "recovery", "outcome"}
    for f in feature_cols:
        for tok in outcome_tokens:
            if tok in f.lower() and not f.lower().startswith("combine"):
                raise AssertionError(f"potential target leakage: feature '{f}' "
                                     f"contains outcome token '{tok}'")


def _pair(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    return x[m], y[m]

## 2. Tracking data quality (Phase 2)

In [ ]:
QUALITY_REPORT = None
if DATA_PRESENT:
    ct = pl.read_csv(DATA_DIR / "combine_tracking.csv", infer_schema_length=100000, null_values=["", "NA", "N/A", "na", "n/a", "null", "NULL", "None", "NaN"])
    # normalise column names (snake/camel drift guard)
    ct = ct.rename({c: re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", c).lower() for c in ct.columns})
    drill_col = next((c for c in ("drill_name", "drill", "drill_type") if c in ct.columns), None)
    attempt_col = next((c for c in ("attempt", "attempt_id", "run") if c in ct.columns), None) or "attempt"
    if attempt_col not in ct.columns:
        ct = ct.with_columns(pl.lit(1).alias("attempt"))
    q_rows = []
    for (nid, drill, att), g in ct.group_by(["nfl_id", drill_col, attempt_col], maintain_order=True):
        if "frame_id" not in g.columns:
            g = g.with_columns(pl.int_range(pl.len()).over("nfl_id").alias("frame_id"))
        g = g.sort("frame_id")
        x = g["x"].to_numpy(); y = g["y"].to_numpy()
        t = np.arange(len(g)) * 0.1
        v = attempt_quality_verdict(x, y, t, domain="combine")
        q_rows.append({"nfl_id": nid, "drill": canonical_drill_nb(drill), "attempt": att,
                       "flags": v["flags"], "usable": v["usable"]})
    qq = pd.DataFrame(q_rows)
    QUALITY_REPORT = {
        "n_attempts": len(qq), "n_usable": int(qq.usable.sum()),
        "share_usable": round(qq.usable.mean(), 4),
        "flag_counts": qq.explode("flags")["flags"].value_counts().to_dict(),
    }
    print(json.dumps(QUALITY_REPORT, indent=2, default=str))
else:
    print("GATED: combine_tracking.csv not present -- no quality claims.")

## 3. Mirror Response Index extraction (Phase 3)

Every attempt of an MRI-eligible drill (shuttles, three-cone, position drills)
is quality-gated, then passed through `attempt_components` (the parity-tested
implementation above). Attempts are aggregated to players; the composite is
standardised within position x draft-year cohorts.

In [ ]:
MRI_PLAYERS = []
if DATA_PRESENT:
    players = pl.read_csv(DATA_DIR / "players.csv", infer_schema_length=100000, null_values=["", "NA", "N/A", "na", "n/a", "null", "NULL", "None", "NaN"])
    players = players.rename({c: re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", c).lower() for c in players.columns})
    pinfo = {str(r["nfl_id"]): r for r in players.to_dicts()}
    feature_rows = []
    for (nid, drill, att), g in ct.group_by(["nfl_id", drill_col, attempt_col], maintain_order=True):
        d = canonical_drill_nb(drill)
        if d not in MRI_ELIGIBLE_DRILLS:
            continue
        if "frame_id" not in g.columns:
            g = g.with_columns(pl.int_range(pl.len()).over("nfl_id").alias("frame_id"))
        g = g.sort("frame_id")
        x, y = g["x"].to_numpy(), g["y"].to_numpy()
        t = np.arange(len(g)) * 0.1
        v = attempt_quality_verdict(x, y, t, domain="combine")
        if not v["usable"]:
            continue
        f = attempt_components(x, y, 0.1)
        fd = f.to_dict(); fd.update({"nfl_id": str(nid), "drill": d, "attempt": att})
        feature_rows.append(fd)
    fdf = pd.DataFrame(feature_rows)
    if fdf.empty:
        print("No usable MRI-eligible attempts found -- check drill labels and quality flags.")
    else:
        print(f"MRI-eligible usable attempts: {len(fdf)} | players: {fdf.nfl_id.nunique()}")
    agg = fdf.groupby("nfl_id").agg(
        n_attempts=("attempt", "count"),
        reorientation_latency=("reorientation_latency", "mean"),
        turn_speed_retention=("turn_speed_retention", "mean"),
        directional_transition_efficiency=("directional_transition_efficiency", "mean"),
        exit_acceleration=("exit_acceleration", "mean")).reset_index() if not fdf.empty else None
    if agg is not None:
        for c in agg.columns[2:]:
            agg[c] = agg[c].astype(float)
        agg["position_group"] = agg.nfl_id.map(lambda i: pinfo.get(i, {}).get("position", "?"))
    # composite within cohorts (robust z, latency flipped)
    comp_cols = ["reorientation_latency", "turn_speed_retention",
                 "directional_transition_efficiency", "exit_acceleration"]
    def composite(gr):
        z = {c: robust_z(gr[c].to_numpy()) * (-1.0 if c == "reorientation_latency" else 1.0)
             for c in comp_cols}
        used, num, den = [], 0.0, 0.0
        for i in range(len(gr)):
            u, n_, d_ = [], 0.0, 0.0
            for c in comp_cols:
                if np.isfinite(z[c][i]):
                    u.append(c); n_ += z[c][i]; d_ += 1.0
            used.append(u)
            num, den = n_, d_
        gr = gr.copy()
        gr["mri_components_used"] = used
        gr["mri_composite"] = [ (num/den if len(u) >= RP["min_components_for_composite"] else np.nan)
                                for u in used ]
        return gr
    if agg is not None:
        MRI_PLAYERS = pd.concat([composite(g) for _, g in agg.groupby("position_group")]).to_dict("records")
        mdf = pd.DataFrame(MRI_PLAYERS)
        print(mdf[["nfl_id", "position_group", "n_attempts", "mri_composite"]].head(10).to_string(index=False))
        print(f"\nPlayers with a composite (>= {RP['min_components_for_composite']} components): "
              f"{mdf.mri_composite.notna().sum()} / {len(mdf)}")
else:
    print("GATED: competition data not present -- no MRI extraction on real athletes.")

## 4. NFL coverage movement analytics (Phase 4)

Regular-season pass plays only (`games.season_type == 'REG'`). A play is a pass
play for this analysis when the defender carries a non-null, non-`NO_ASSIGNMENT`
`coverage_assignment` (the 2027 file has no is_pass_play flag; `separation_at_pass_forward`
is populated receiver-side only). The DB's associated receiver is the
**nearest eligible receiver at the pass-forward frame** (proximity approximation,
flagged). The primary outcome is **tracking-derived separation at the
pass-forward frame** — the host-provided `separation_at_pass_forward` column is
reported as a cross-check where both sides have it, never hand-filled.
Tracking files carry no `frame_id` column; the frame index is reconstructed per
(play, entity) at 10 Hz from row order. DuckDB scans are batched **per game**
(697 games x ~0.04 s zonemap scan on a persistent, sorted database), never one
CSV scan per play.

In [ ]:
COVERAGE_PLAYERS, COVERAGE_LOG = [], {}
if DATA_PRESENT:
    games = pl.read_csv(DATA_DIR / "games.csv", infer_schema_length=100000, null_values=["", "NA", "N/A", "na", "n/a", "null", "NULL", "None", "NaN"])
    games = games.rename({c: re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", c).lower() for c in games.columns})
    stg = "season_type" if "season_type" in games.columns else "game_type"
    reg_ids = set(games.filter(pl.col(stg) == "REG")["game_id"].to_list())
    pp = pl.read_csv(DATA_DIR / "player_play.csv", infer_schema_length=100000, null_values=["", "NA", "N/A", "na", "n/a", "null", "NULL", "None", "NaN"])
    pp = pp.rename({c: re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", c).lower() for c in pp.columns})
    pp = pp.filter(pl.col("game_id").is_in(list(reg_ids)))
    pos_col = "lined_up_position" if "lined_up_position" in pp.columns else "position"
    sep_col = "separation_at_pass_forward" if "separation_at_pass_forward" in pp.columns else None
    DBS = {"CB", "S", "DB", "FS", "SS", "SLOT_CB", "HIGH_SAFETY", "FREE_SAFETY",
           "STRONG_SAFETY", "NICKEL", "DIME", "SLOT_CORNER"}
    DB_PLAYS = pp.filter(pl.col(pos_col).str.to_uppercase().is_in(list(DBS)))
    # pass play for a defender = a non-null, non-NO_ASSIGNMENT coverage assignment
    cov_col = next((c for c in ("coverage_assignment", "coverage_assignment_at_snap")
                    if c in pp.columns), None)
    if cov_col:
        PASS_PLAYS = DB_PLAYS.filter(pl.col(cov_col).is_not_null()
                                      & ~pl.col(cov_col).str.to_uppercase().eq("NO_ASSIGNMENT"))
    else:
        PASS_PLAYS = DB_PLAYS
    RECVS = {"WR", "TE", "RB", "FB", "HB", "SLOT_WR", "SLOT", "X", "Z", "H", "Y"}
    recv = pp.filter(pl.col(pos_col).str.to_uppercase().is_in(list(RECVS)))
    print(f"REG games: {len(reg_ids)} | REG plays: {pp.height} | DB rows: {DB_PLAYS.height} | pass reps: {PASS_PLAYS.height}")

    # Persistent sorted DuckDB over the tracking files (built once; per-game zonemap reads)
    gt_files = [f for f in (DATA_DIR / f"game_tracking_{y}.csv" for y in (2023, 2024, 2025)) if f.exists()]
    db_path = Path("./tracking.duckdb")
    if not db_path.exists():
        con = duckdb.connect(str(db_path))
        con.execute("SET threads TO (" + str(os.cpu_count() or 2) + ")")
        union = " UNION ALL ".join(
            f"SELECT * FROM read_csv_auto('{f.as_posix()}', sample_size=100000, all_varchar=false)"
            for f in gt_files)
        con.execute(f"CREATE TABLE gt_all AS SELECT * FROM ({union}) ORDER BY game_id, play_id, nfl_id, time")
        con.execute("CHECKPOINT"); con.close()
    con = duckdb.connect(str(db_path), read_only=True)

    by_game = {}
    for r in PASS_PLAYS.select(["game_id", "play_id", "nfl_id"]).unique().to_dicts():
        by_game.setdefault(r["game_id"], []).append(r)
    RECV_KEEP = ["game_id", "play_id", "nfl_id"] + ([sep_col] if sep_col else [])
    RECV_KEEP = [c for c in RECV_KEEP if c in recv.columns]
    recv_by_play = {}
    for r in recv.select(RECV_KEEP).to_dicts():
        recv_by_play.setdefault((r["game_id"], r["play_id"]), []).append(r)
    play_rows, unresolved = [], 0
    for i, (gid, rows) in enumerate(sorted(by_game.items()), 1):
        fr = con.execute("SELECT * FROM gt_all WHERE game_id = ? ORDER BY play_id, nfl_id, time", [gid]).fetchdf()
        if len(fr) == 0:
            unresolved += len(rows); continue
        fr["frame_id"] = fr.groupby(["play_id", "nfl_id"]).cumcount()
        for drow in rows:
            dfr = fr[(fr.play_id == drow["play_id"]) & (fr.nfl_id == drow["nfl_id"])]
            if len(dfr) < 5:
                unresolved += 1; continue
            x, y = dfr.x.to_numpy(), dfr.y.to_numpy()
            ev = dfr.event.tolist() if "event" in fr else []
            pf = next((j for j, e in enumerate(ev) if e and "forward" in str(e).lower()), None)
            if pf is None:
                unresolved += 1; continue
            # eligible receivers on this play
            rr = [r for r in recv_by_play.get((gid, drow["play_id"]), [])]
            recv_id, method = None, "unresolved_proximity"
            if rr:
                # distance at the pass-forward frame (approximate: receiver and
                # defender sampled at the same frame index)
                d_best, d_best_id = np.inf, None
                for r in rr:
                    rfr = fr[(fr.play_id == drow["play_id"]) & (fr.nfl_id == r["nfl_id"])]
                    if len(rfr) and pf < min(len(x), len(rfr)):
                        d = float(np.hypot(x[pf] - rfr.x.iloc[pf], y[pf] - rfr.y.iloc[pf]))
                        if d < d_best:
                            d_best, d_best_id = d, r["nfl_id"]
                if d_best_id is not None:
                    recv_id, method = d_best_id, "nearest_proximity"
            rec = {"nfl_id": str(drow["nfl_id"]), "game_id": gid,
                   "play_id": drow["play_id"], "association_method": method,
                   "n_frames": int(len(x))}
            if pf < len(x) - 1:
                # primary outcome: tracking-derived separation at pass forward
                if recv_id is not None:
                    rfr = fr[(fr.play_id == drow["play_id"]) & (fr.nfl_id == recv_id)]
                    if len(rfr) and pf < min(len(x), len(rfr)):
                        rec["sep_at_pf"] = float(np.hypot(x[pf] - rfr.x.iloc[pf], y[pf] - rfr.y.iloc[pf]))
                # cross-check: host separation (receiver-side; NA for defenders)
                host = [r for r in rr if str(r["nfl_id"]) == str(drow["nfl_id"])]
                if host and sep_col:
                    v = host[0].get(sep_col)
                    if v is not None and pd.notna(v):
                        rec["sep_at_pf_host"] = float(v)
            if "sep_at_pf" in rec:
                play_rows.append(rec)
    con.close()
    print(f"Replayed {len(by_game)} games | tracking-resolved reps: {len(play_rows)} | unresolved: {unresolved}")
    prdf = pd.DataFrame(play_rows)
    if len(prdf):
        cov = prdf.groupby("nfl_id").agg(
            n_coverage_plays=("play_id", "count"),
            sep_at_pass_forward_mean=("sep_at_pf", "mean"),
            sep_p90=("sep_at_pf", lambda s: np.quantile(s, 0.9)),
            open_rate_3yd=("sep_at_pf", lambda s: float((s >= RP["open_threshold_yd"]).mean())),
        ).reset_index()
        cov["meets_min_exposure"] = cov.n_coverage_plays >= RP["min_coverage_plays"]
        COVERAGE_PLAYERS = cov.to_dict("records")
        COVERAGE_LOG["association_counts"] = prdf.association_method.value_counts().to_dict()
        COVERAGE_LOG["n_plays"] = len(prdf)
        COVERAGE_LOG["unresolved"] = int(unresolved)
        print(f"Coverage plays with outcome: {len(prdf)} | DBs: {cov.nfl_id.nunique()}")
        print(f"Association methods: {COVERAGE_LOG['association_counts']}")
        print(cov.head(10).to_string(index=False))
else:
    print("GATED: competition data not present -- no coverage analytics.")

## 5. Statistical validation (Phase 5, pre-registered)

Model A = traditional Combine | Model B = A + MRI | Model C = B + context.
Player-grouped cross-validation, player-clustered bootstrap, permutation
inference, leave-one-cohort-out, BH-FDR on secondary components.

In [ ]:
RESEARCH = None
if DATA_PRESENT and len(MRI_PLAYERS) and len(COVERAGE_PLAYERS):
    mri_df = pd.DataFrame(MRI_PLAYERS)[["nfl_id", "mri_composite", "n_attempts"]]
    cov_df = pd.DataFrame(COVERAGE_PLAYERS)
    df = mri_df.merge(cov_df, on="nfl_id")
    df = df[df.meets_min_exposure]
    # traditional combine (best attempt per drill)
    cr = pl.read_csv(DATA_DIR / "combine_results.csv", infer_schema_length=100000, null_values=["", "NA", "N/A", "na", "n/a", "null", "NULL", "None", "NaN"])
    cr = cr.rename({c: c.lower() for c in cr.columns})
    cr = cr.rename({c: f"combine_{c}" for c in cr.columns if c != "nfl_id"})
    trad = cr.to_pandas()
    trad["nfl_id"] = trad["nfl_id"].astype(str)
    df = df.merge(trad, on="nfl_id", how="left")
    print(f"Analysis sample: {len(df)} DBs (exposure >= {RP['min_coverage_plays']} plays)")
    if len(df) >= 10 and df.mri_composite.notna().sum() >= 8:
        y = df.sep_at_pass_forward_mean.to_numpy(float)
        mri_c = df.mri_composite.to_numpy(float)
        forty = pd.to_numeric(df.get("combine_forty"), errors="coerce").to_numpy(float)
        shuttle = pd.to_numeric(df.get("combine_short_shuttle"), errors="coerce").to_numpy(float)
        def med_impute(a):
            a = a.copy(); a[~np.isfinite(a)] = np.nanmedian(a); return a
        forty_i, shuttle_i = med_impute(forty), med_impute(shuttle)
        mri_i = med_impute(mri_c)
        players = df.nfl_id.to_numpy()
        assert_no_leakage(["mri_composite"], ["sep_at_pass_forward_mean"])
        stat = lambda xv, yv: partial_spearman(xv, yv, np.column_stack([forty_i, shuttle_i]))["rho"]
        RESEARCH = {
            "n_players": int(len(df)),
            "partial_spearman": partial_spearman(mri_c, y, np.column_stack([forty_i, shuttle_i])),
            "permutation": permutation_test(stat, mri_c, y, n_perm=2000),
            "bootstrap_ci": clustered_bootstrap_ci(
                lambda idx: spearman(mri_c[idx], y[idx])["rho"], players, n_boot=500),
            "models_oos": {
                "A_traditional": cross_validated_oos(y, np.column_stack([forty_i, shuttle_i]), players),
                "B_plus_mri": cross_validated_oos(y, np.column_stack([forty_i, shuttle_i, mri_i]), players),
            },
        }
        print(json.dumps(RESEARCH, indent=2, default=str))
    else:
        print("Sample below pre-registered minimum (n<10 or <8 players with MRI) -> no inference.")
else:
    print("GATED: upstream tables absent -- no statistical claims.")

## 6. Figures (all from the canonical analysis results above)

Every figure is produced from the same tables this notebook computed. Figures
that require competition data are skipped (with an explicit note) when the
dataset is not attached. Figure 1 (pipeline schematic) and the labelled
synthetic demonstration always render.

In [ ]:
FIGURES = {}

def fig_pipeline():
    fig, ax = plt.subplots(figsize=(10, 3.2))
    stages = ["RAW 9 CSVs\n(audited, checksummed)", "QUALITY\n(frame-level checks)",
              "MRI FEATURES\n(5 components)", "COVERAGE ANALYTICS\n(REG plays only)",
              "STATISTICAL VALIDATION\n(A/B/C, bootstrap, permutation)"]
    for i, s in enumerate(stages):
        ax.add_patch(plt.Rectangle((i * 2, 0), 1.7, 1.4, fill=False, lw=1.6))
        ax.text(i * 2 + 0.85, 0.7, s, ha="center", va="center", fontsize=9)
        if i < len(stages) - 1:
            ax.annotate("", xy=(i * 2 + 1.95, 0.7), xytext=(i * 2 + 1.72, 0.7),
                        arrowprops=dict(arrowstyle="->", lw=1.4))
    ax.set_xlim(-0.1, len(stages) * 2); ax.set_ylim(-0.1, 1.6); ax.axis("off")
    ax.set_title("Figure 1. SHADOWLINE analytical pipeline (data-gated at every stage)")
    plt.tight_layout(); plt.show()
    FIGURES["fig1_pipeline"] = "rendered (schematic)"

def fig_demo():
    fx = synthetic_shuttle_attempt()
    x, y = np.array(fx["x"]), np.array(fx["y"])
    sp = np.array(fx["speed"]); t = np.array(fx["t"])
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    axes[0].plot(x, y, "-", color="#0a7d5f", lw=1.8)
    for e in detect_reversals(np.array(fx["heading"]), sp, fx["dt"]):
        axes[0].plot(x[e["centre_idx"]], y[e["centre_idx"]], "o", color="#c2410c", ms=9, zorder=5)
    axes[0].set_xlabel("x (yd)"); axes[0].set_ylabel("y (yd)")
    axes[0].set_title("Synthetic shuttle trajectory + detected plants")
    axes[1].plot(t, sp, color="#1f2937", lw=1.4)
    for e in detect_reversals(np.array(fx["heading"]), sp, fx["dt"]):
        axes[1].axvline(t[e["centre_idx"]], color="#c2410c", ls="--", lw=1)
    axes[1].set_xlabel("time (s)"); axes[1].set_ylabel("speed (yd/s)")
    axes[1].set_title("Speed profile with plant windows")
    fig.suptitle("Figure 2 (DEMONSTRATION). MRI algorithm on SYNTHETIC fixture — NOT NFL data",
                 fontsize=11, color="#9a3412")
    plt.tight_layout(); plt.show()
    FIGURES["fig2_synthetic_demo"] = "rendered (labelled synthetic)"

fig_pipeline(); fig_demo()

In [ ]:
def fig_real(figkey, fn):
    if not (DATA_PRESENT and len(MRI_PLAYERS)):
        print(f"{figkey}: SKIPPED -- requires competition data (not attached).")
        FIGURES[figkey] = "skipped_no_data"
        return
    try:
        fn()
        FIGURES[figkey] = "rendered"
    except Exception as exc:
        print(f"{figkey}: failed -> {exc}")
        FIGURES[figkey] = f"error: {exc}"

if DATA_PRESENT and len(MRI_PLAYERS):
    mdf = pd.DataFrame(MRI_PLAYERS)
    # Fig 3: MRI component distributions
    def _f3():
        comps = ["reorientation_latency", "turn_speed_retention",
                 "directional_transition_efficiency", "exit_acceleration"]
        fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
        for ax, c in zip(axes, comps):
            vals = mdf[c].dropna()
            ax.hist(vals, bins=min(25, max(8, len(vals) // 4)), color="#0a7d5f", alpha=.85)
            ax.set_title(c.replace("_", " "), fontsize=9)
            ax.set_xlabel({"reorientation_latency": "s", "turn_speed_retention": "ratio",
                           "directional_transition_efficiency": "ratio",
                           "exit_acceleration": "yd/s²"}[c])
            ax.set_ylabel(f"players (n={len(vals)})")
        fig.suptitle(f"Figure 3. MRI component distributions (n={len(mdf)} players, all cohorts)")
        plt.tight_layout(); plt.show()
    fig_real("fig3_mri_distributions", _f3)
    # Fig 5: combine-to-game relationship
    def _f5():
        if len(COVERAGE_PLAYERS) == 0:
            print("fig5: no coverage players"); return
        d = pd.DataFrame(MRI_PLAYERS).merge(pd.DataFrame(COVERAGE_PLAYERS), on="nfl_id")
        d = d[d.meets_min_exposure]
        fig, ax = plt.subplots(figsize=(6.2, 4.4))
        sc = ax.scatter(d.mri_composite, d.sep_at_pass_forward_mean,
                        c=d.n_coverage_plays, cmap="viridis", s=55, edgecolor="white")
        if len(d) >= 4:
            rho, p = sps.spearmanr(d.mri_composite, d.sep_at_pass_forward_mean, nan_policy="omit")
            ax.set_title(f"Figure 5. MRI vs mean separation at pass-forward\n"
                         f"Spearman rho={rho:.2f}, p={p:.3f}, n={len(d)} DBs")
        plt.colorbar(sc, label="coverage plays")
        ax.set_xlabel("MRI composite (cohort z)"); ax.set_ylabel("mean separation (yd)")
        plt.tight_layout(); plt.show()
    fig_real("fig5_combine_to_game", _f5)
    # Fig 7: baseline vs enhanced model
    def _f7():
        if not RESEARCH or "models_oos" not in (RESEARCH or {}):
            print("fig7: research results not available"); return
        a = RESEARCH["models_oos"]["A_traditional"].get("oos_spearman")
        b = RESEARCH["models_oos"]["B_plus_mri"].get("oos_spearman")
        if a is None or b is None:
            print("fig7: OOS metrics unavailable (insufficient n)"); return
        fig, ax = plt.subplots(figsize=(5.2, 3.8))
        ax.bar(["Model A\n(traditional)", "Model B\n(+MRI)"], [a, b],
               color=["#9ca3af", "#0a7d5f"], width=.55)
        ax.set_ylabel("out-of-sample Spearman (player-grouped 5-fold)")
        ax.set_title("Figure 7. Baseline vs enhanced model")
        for i, v in enumerate([a, b]):
            ax.text(i, v + .01, f"{v:.3f}", ha="center")
        plt.tight_layout(); plt.show()
    fig_real("fig7_model_comparison", _f7)
print(json.dumps(FIGURES, indent=2))

## 7. Findings, limitations and citations

**Status.** This notebook is deterministic (fixed seeds), executes top-to-bottom,
and hard-gates every competition finding on data presence. When the dataset is
attached, the Findings section below reports exactly what the pre-registered
tests produced — including null results, which are reported as nulls.

**Limitations (pre-declared).**
1. *Exposure bias*: 13 of 510 players never played an NFL snap; low exposure
   is treated as missing information, never as poor ability. The primary
   analysis requires ≥ 20 coverage plays and reports exclusions.
2. *Association quality*: receiver association uses targeting when available
   and a flagged proximity approximation otherwise; unresolved associations
   are counted and reported.
3. *Cohort measurement differences*: 2023-2025 Combine tracking may differ in
   vendor configuration; distribution shifts are tested before pooling and
   reported.
4. *Small-n inference*: the eligible-DB sample is modest; inference relies on
   robust rank statistics, clustered resampling and permutation tests rather
   than asymptotic normality.
5. *Composite validity*: the MRI composite is not assumed valid. Model B vs A
   improvement is the pre-registered criterion; component-level findings are
   FDR-controlled.

**Data.** NFL Big Data Bowl 2027 (Kaggle), CC BY-NC 4.0 — competition data is
not redistributed by this notebook; attach the official competition dataset.

**Citations.** NFL Football Operations & Kaggle, *NFL Big Data Bowl 2027:
Combine to NFL Performance* (2026). Zebra Technologies optical/radio tracking
at 10 Hz, 2023-2025 Combines, 510 rookies, 6,310 drill attempts, 1,002 games.